# Serial Communication Code Cards
## Try me
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ffraile/computer_science_tutorials/blob/main/source/Data%20Manipulation/exercises/Serial%20Communication%20cards.ipynb)[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/ffraile/computer_science_tutorials/main?labpath=source%2FData%20Manipulation%2Fexercises%2FSerial%20Communication%20cards.ipynb)

## How to use
- Each card mirrors an A4 classroom prompt. **Predict first** (or discuss), then run the cell to check.
- Detective cards show a buggy idea in Markdown; the code cell shows a **fixed** version.
- Keep explanations short and schematic (*what* → *why*).

### Turn Gemini into a coding tutor (no direct answers)
Paste this in your first chat with Gemini to keep it in “tutor mode”:

```markdown
You are a **coding tutor** for Python in Jupyter/Colab. Follow the **course motto** “do not give up learning.”

### Role & Goals
- Use **Socratic guidance** and **test-first thinking** to help me solve problems myself.
- Help me read errors, reason about state, and make small, safe iterations.

### Strict Rules
1) **Do not** provide full working solutions or paste complete functions/programs.
   - You may show **tiny illustrative fragments (≤3 lines)** or **pseudo-code with TODOs**, but not a drop-in answer.
2) Prefer **questions over answers**; offer **one small next step** at a time.
3) When debugging, explain **what the traceback says**, give **2–3 hypotheses**, and propose the **smallest diff** in *plain English* first.
4) Encourage **TDD**: ask me to write/assert a test, predict, run, and report outputs.
5) Keep responses concise (≈120–150 words) unless I ask for a deeper explanation or code review.
6) Ask me to **run code and share results**; adapt based on the output.
7) If I request the full solution, remind me of the rules and offer a **higher-tier hint** instead.
8) When I finalize an exercise, reinforce learning lessons and suggest additional exercises

### Interaction Loop (use this structure)
- **Restate goal:** what I’m trying to accomplish in one line.
- **Diagnose:** key assumption to check or error to interpret.
- **Hint (tiered):**
  - Tier 1: Conceptual nudge (no code).
  - Tier 2: Directed hint (identify line/construct to change).
  - Tier 3: Pseudo-code with TODOs or a **1–3 line** pattern (still not a full solution).
- **Next action:** one concrete step for me to try now.
- **Ask back:** what to run/paste (output, test result, or traceback).

### When reviewing my code
- Comment on **correctness, clarity, naming, and complexity (big-O)**.
- Suggest **tests** I’m missing (boundaries, empty cases, error paths).

### Safety & Ethics
- No secrets or private data in prompts.
- avoid library functions/APIs unless I ask.

Stay in tutor mode for the whole session.
```


## Code Cards

1. The following function is used in Python to log continous data from an Arduino device. Explain in your own words why each call to the function ```open``` is made with different modes ('w' and 'a+'). What would happen if you used 'a+' in both cases?

```python
def log_continuous(ser, out_path="arduino_log.csv", period=1.0, sim_mode=False):
    """
    Poll metrics every 'period' seconds and append to CSV.
    Stop with Ctrl+C.
    """
    fields = ["timestamp", "uptime_ms", "vcc_mv", "led"]
    # Ensure output directory exists
    if not os.path.exists(os.path.dirname(out_path)):
        with open(out_path, 'w', newline='', encoding='utf-8') as csvfile:
            writer = csv.DictWriter(csvfile, fieldnames=fields)
            writer.writeheader()
    print(f"📝 Logging to {out_path} every {period:.2f}s — press Ctrl+C to stop.")
    try:
        with open(out_path, "a+", encoding="utf-8") as f:
            while True:
                t0 = time.time()
                writer = csv.DictWriter(f, fieldnames=fields)
                row = read_metrics_once(ser, sim_mode=sim_mode)
                writer.writerow(row)
                f.flush()
                # Console status line
                print(row)

                # pacing delay:
                dt = time.time() - t0
                sleep_left = period - dt
                if sleep_left > 0:
                    time.sleep(sleep_left)
    except KeyboardInterrupt:
        print("\n⏹️  Logging stopped.")
```

Given the following Arduino board circuit connected to the computer via USB:

![Arduino Tone Keyboard](img/tone_keyboard.png)

This is the code that runs on the Arduino board:

```cpp
int pos = 0;

void tone(int pin, int frequency, int duration) {
  // Calculate the period of the wave
  int period = 1000000 / frequency; // in microseconds
  int pulse = period / 2; // 50% duty cycle

  // Calculate the number of cycles to play
  int cycles = (frequency * duration) / 1000;

  for (int i = 0; i < cycles; i++) {
    digitalWrite(pin, HIGH);
    delayMicroseconds(pulse);
    digitalWrite(pin, LOW);
    delayMicroseconds(pulse);
  }
}

void setup()
{
  pinMode(A0, INPUT);
  pinMode(8, OUTPUT);
  pinMode(A1, INPUT);
  pinMode(A2, INPUT);
}

void loop()
{
  // if button press on A0 is detected
  if (digitalRead(A0) == HIGH) {
    tone(8, 440, 100);
  }
  // if button press on A1 is detected
  if (digitalRead(A1) == HIGH) {
    tone(8, 494, 100);
  }
  // if button press on A0 is detected
  if (digitalRead(A2) == HIGH) {
    tone(8, 523, 100);
  }
  //Check if serial data is available
  if (Serial.available() > 0) {
    note = Serial.read(); // read the incoming character
    switch (note) {
      case 'A':
        tone(8, 440, 100);
        break;
      case 'B':
        tone(8, 494, 100);
        break;
      case 'C':
        tone(8, 523, 100);
        break;
    }
  }
}
```

And this is the Python code:

```python
import serial
import time
import sys
notes = [{'command': 'A', 'name': 'A4', 'tone': 57},
         {'command': 'B', 'name': 'B4', 'tone': 59},
         {'command': 'C', 'name': 'C5', 'tone': 60}]

def print_menu():
    print("Available notes:")
    for note in notes:
        print(f"Press {note[command]} to play note {note['name']} (Tone {note['tone']})")

    print("Press 0 to exit")

def main():
    # Set up serial connection (adjust 'COM3' and baudrate as needed)
    ser = serial.Serial('COM3', 9600, timeout=1)
    time.sleep(2)  # Wait for the connection to establish

    while True:
        print_menu()
        choice = input("Enter your choice: ")

        if choice not in [note['command'] for note in notes] + ['0']:
            print("Invalid choice. Please try again.")
            continue

        if choice == '0':
            print("Exiting...")
            break


        for note in notes:
            if choice == note['command']:
                ser.write(note['command'].encode('utf-8'))
                print(f"Playing note {note['name']} ({note['tone']})")
                break

    ser.close()
```
Answer the following questions:
1. What is the sequence of frequencies played when the user inputs 'A', 'C', 'B' in that order?
2. Why is comprehension used in the line ```if choice not in [note['command'] for note in notes] + ['0']:``?
3. What changes would you make to the Python code to add a new note 'D5' with command 'D' (tone 61)? What needs to be added to the Arduino code to support this new note (frequency of new note is 587 Hz)?

3. The following Arduino project connects a potentiometer, a simple mechanical device that provides a varying resistance based on the position of a knob. This basic set up is used to illustrate how smoothing averages can be computed from noisy sensor data. A smoothing average helps to reduce the effect of random fluctuations in the sensor readings, by averaging multiple readings over time.

![Smoothing Average with Potentiometer](img/smoothing_average.png)

The Arduino code reads the potentiometer value (an integer between 0 and 1023) and sends it over serial communication to a connected computer. The Python code reads these values, computes a smoothing average, and plots the results in real-time.

This is the Arduino code:

```cpp
int inputPin = A0;

void setup() {
  // initialize serial communication with computer:
  Serial.begin(9600);
  // initialize all the readings to 0:
  for (int thisReading = 0; thisReading < numReadings; thisReading++) {
    readings[thisReading] = 0;
  }
}

void loop() {
  // read the potentiometer:
  int sensorValue = analogRead(inputPin);
  // send the value to the computer:
  Serial.print("pot_value: ");
  Serial.println(sensorValue);
  // wait 100 milliseconds before the next reading:
  delay(100);
}
```

And this is the Python code:

```python
import serial
import time
import matplotlib.pyplot as plt

def read_potentiometer(ser):
    line = ser.readline().decode('utf-8').strip()
    if line.startswith("value:"):
        value_str = line.split(":")[0].strip()
        return int(value_str)
    return None
def main():
    ser = serial.Serial('COM3', 9600, timeout=1)
    time.sleep(2)  # Wait for the connection to establish
    average_size = 10
    readings = [0 for i in range(average_size)]
    plt.ion()    # Turn on interactive mode
    fig, ax = plt.subplots()
    line1, = ax.plot([], [], 'b-', label='Raw Data')
    line2, = ax.plot([], [], 'r-', label='Smoothing Average')
    ax.set_xlabel('measurement')
    ax.set_ylabel('Value')
    ax.set_xticks(range(10))
    ax.set_yticks(range(0, 1023, 50))
    ax.set_xlim(0, 9)
    ax.set_ylim(0, 1023)
    ax.legend()
    # show grid

    pos = 0
    while True:
        pot_value = read_potentiometer(ser)
        if pot_value is not None:
            readings[pos%average_size] = pot_value
            pos += 1
            if len(readings) > average_size:
                readings.pop(0)
            smoothing_avg = sum(readings) / len(readings)

            # Update plot
            line1.set_xdata(range(len(readings)))
            line1.set_ydata(readings)
            line2.set_xdata(range(len(readings)))
            line2.set_ydata([smoothing_avg] * len(readings))
            plt.draw()
            plt.pause(0.1)
    ser.close()
if __name__ == "__main__":
    main()
```
Answer the following questions:
1. Fix the bugs in the `read_potentiometer` function so that it correctly extracts the potentiometer value from the serial input. What changes are needed?
2. Explain how the smoothing average is calculated in the Python code. What is the purpose of the `window_size` variable?
3. The 11 first values of the potentiometer readings are ```850, 900, 950, 900, 850, 900, 950, 1000, 950, 900, 1000```. Use these values to manually compute the value of the ```readings``` list and the smoothing average after processing all 11 readings, and plot the expected output in the figure below.

![Expected Smoothing Average Plot](img/smoothing_average_plot.png)

3. Identify and explain any potential issues with the way the `readings` list is managed in the Python code. Hint: What happens when less than `window_size` readings are collected? Can you suggest a fix or alternative approach?